# Assignment 04 - Integrated Field Mapping

This notebook builds an integrated spatial map using field boundaries and NRCS soil data, verifies CRS alignment, and exports a dashboard-ready PNG asset.

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import contextily as ctx
from matplotlib.patches import Patch

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root from current working directory")

PROJECT_ROOT = find_project_root(Path.cwd())
BOUNDARIES_PATH = PROJECT_ROOT / "data" / "boundaries" / "nm_top_200_fields.geojson"
SOIL_PATH = PROJECT_ROOT / "data" / "soil" / "nm_soil_data.csv"
OUTPUT_PATH = PROJECT_ROOT / "output" / "dashboard_assets" / "field_spatial_map.png"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

BOUNDARIES_PATH, SOIL_PATH, OUTPUT_PATH

In [ ]:
fields_gdf = gpd.read_file(BOUNDARIES_PATH)
soil_df = pd.read_csv(SOIL_PATH)

dominant_soil = (
    soil_df.sort_values(["field_id", "comppct_r"], ascending=[True, False])
    .drop_duplicates(subset=["field_id"])
    [["field_id", "compname", "drainagecl", "muname", "comppct_r"]]
    .rename(
        columns={
            "compname": "dominant_soil_type",
            "drainagecl": "dominant_drainage",
            "muname": "map_unit_name",
            "comppct_r": "dominant_component_pct",
        }
    )
)

soil_spatial_gdf = fields_gdf.merge(dominant_soil, on="field_id", how="left")
soil_spatial_gdf = gpd.GeoDataFrame(soil_spatial_gdf, geometry="geometry", crs=fields_gdf.crs)

print(f"Field boundaries CRS: {fields_gdf.crs}")
print(f"Soil spatial layer CRS (before): {soil_spatial_gdf.crs}")

if fields_gdf.crs != soil_spatial_gdf.crs:
    soil_spatial_gdf = soil_spatial_gdf.to_crs(fields_gdf.crs)
    print("CRS mismatch detected and fixed with .to_crs()")
else:
    print("CRS already aligned; no transformation needed.")

print(f"Soil spatial layer CRS (after): {soil_spatial_gdf.crs}")

In [ ]:
top_n = 12
top_soils = soil_spatial_gdf["dominant_soil_type"].value_counts().head(top_n).index
plot_gdf = soil_spatial_gdf.copy()
plot_gdf["soil_group"] = plot_gdf["dominant_soil_type"].where(plot_gdf["dominant_soil_type"].isin(top_soils), "Other soils")
plot_gdf["soil_group"] = plot_gdf["soil_group"].fillna("Unknown")

us_counties = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2024/COUNTY/tl_2024_us_county.zip")
nm_counties_all = us_counties[us_counties["STATEFP"] == "35"].copy()
tx_counties_all = us_counties[us_counties["STATEFP"] == "48"].copy()
aoi_names = ["Lea", "Roosevelt", "Curry"]
aoi_counties = nm_counties_all[nm_counties_all["NAME"].isin(aoi_names)].copy()

plot_gdf_3857 = plot_gdf.to_crs(epsg=3857)
nm_counties_3857 = nm_counties_all.to_crs(epsg=3857)
tx_counties_3857 = tx_counties_all.to_crs(epsg=3857)
aoi_counties_3857 = aoi_counties.to_crs(epsg=3857)

minx, miny, maxx, maxy = aoi_counties_3857.total_bounds
context_pad_west = 90000
context_pad_east = 140000
context_pad_ns = 90000

from shapely.geometry import box
context_box = box(
    minx - context_pad_west,
    miny - context_pad_ns,
    maxx + context_pad_east,
    maxy + context_pad_ns,
)

nm_context = nm_counties_3857[nm_counties_3857.geometry.intersects(context_box)].copy()
tx_context = tx_counties_3857[tx_counties_3857.geometry.intersects(context_box)].copy()
context_counties = pd.concat([nm_context, tx_context], ignore_index=True)

categories = sorted(plot_gdf_3857["soil_group"].dropna().unique().tolist())
cmap = plt.get_cmap("tab20")
color_map = {category: cmap(i % 20) for i, category in enumerate(categories)}

fig, ax = plt.subplots(figsize=(25, 11))

cx_min, cy_min, cx_max, cy_max = context_counties.total_bounds
pad_x = (cx_max - cx_min) * 0.012
pad_y = (cy_max - cy_min) * 0.025
ax.set_xlim(cx_min - pad_x, cx_max + pad_x)
ax.set_ylim(cy_min - pad_y, cy_max + pad_y)

ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, zoom=9)

context_counties.boundary.plot(
    ax=ax,
    color="#e5e7eb",
    linewidth=0.65,
    alpha=0.9,
    zorder=2,
)

plot_gdf_3857.plot(
    ax=ax,
    legend=False,
    color=plot_gdf_3857["soil_group"].map(color_map),
    edgecolor="#ffffff",
    linewidth=0.45,
    alpha=0.62,
    zorder=3,
)

aoi_counties_3857.boundary.plot(
    ax=ax,
    color="#fde047",
    linewidth=5.2,
    alpha=0.96,
    zorder=4,
)
aoi_counties_3857.boundary.plot(
    ax=ax,
    color="#7f1d1d",
    linewidth=2.8,
    alpha=1.0,
    zorder=5,
)

county_labels = aoi_counties_3857.copy()
county_labels["label_point"] = county_labels.representative_point()
for _, row in county_labels.iterrows():
    x, y = row["label_point"].x, row["label_point"].y
    ax.text(
        x,
        y,
        row["NAME"],
        fontsize=11,
        fontweight="bold",
        color="#111827",
        ha="center",
        va="center",
        bbox={"facecolor": "#ffffff", "alpha": 0.82, "edgecolor": "#111827", "pad": 2},
        zorder=6,
    )

legend_handles = [
    Patch(facecolor=color_map[category], edgecolor="black", label=category)
    for category in categories
]
legend = ax.legend(
    handles=legend_handles,
    title="Dominant soil type",
    loc="upper right",
    bbox_to_anchor=(0.985, 0.985),
    frameon=True,
    borderaxespad=0.0,
    fontsize=9,
    title_fontsize=10,
)
legend.get_frame().set_alpha(0.95)

ax.annotate(
    "N",
    xy=(0.945, 0.11),
    xytext=(0.945, 0.03),
    xycoords="axes fraction",
    textcoords="axes fraction",
    arrowprops={"facecolor": "black", "edgecolor": "black", "width": 2.2, "headwidth": 10},
    ha="center",
    va="center",
    fontsize=14,
    fontweight="bold",
    zorder=7,
)

ax.text(
    0.015,
    0.02,
    "Area of Interest highlighted in bold: Lea, Roosevelt, and Curry counties",
    transform=ax.transAxes,
    fontsize=10,
    color="#ffffff",
    bbox={"facecolor": "#111827", "alpha": 0.75, "edgecolor": "#fde047", "pad": 4},
    zorder=7,
)

ax.set_title("Integrated Field Spatial Map: AOI Counties with Neighboring New Mexico and Texas Counties", fontsize=16, fontweight="bold", pad=14)
ax.set_axis_off()

plt.tight_layout()
plt.savefig(OUTPUT_PATH, dpi=320, bbox_inches="tight")
plt.show()

print(f"Saved map to: {OUTPUT_PATH}")



## Interpretation

This map overlays dominant NRCS soil types on the three-county area of interest with a tighter extent centered on Lea, Roosevelt, and Curry. The two most abundant dominant soil types remain Kimbrough with 33 fields and Amarillo with 20 fields. Field boundaries and dominant-soil polygons are maintained in WGS84 (EPSG:4326), county boundaries are sourced in NAD83 (EPSG:4269), and all layers are reprojected to Web Mercator (EPSG:3857) for aligned plotting with the satellite basemap. Bold AOI county outlines keep the project focus prominent while preserving enough nearby county geometry for orientation and geographic context.